In [27]:
# Comparación de Modelos Predictivos
## Random Forest, Gradient Boosting, XGBoost

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)


In [29]:
# Cargar datos
df = pd.read_excel('../data/raw/dataset_sintetico_demanda_lima.xlsx')
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

# Features
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['dia'] = df['fecha'].dt.day

for lag in [1, 2, 3, 7]:
    df[f'lag_{lag}'] = df['demanda_real'].shift(lag)

df['media_movil_7'] = df['demanda_real'].rolling(7).mean()
df = df.dropna().reset_index(drop=True)

feature_cols = ['dia_semana', 'mes', 'dia', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'media_movil_7']
X = df[feature_cols].values
y = df['demanda_real'].values

# Dividir datos (80% entrenamiento, 20% prueba)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Datos: {len(df)} días")
print(f"Train: {len(X_train)} días, Test: {len(X_test)} días")
print(f"Features: {len(feature_cols)}")

Datos: 723 días
Train: 578 días, Test: 145 días
Features: 8


In [30]:
# ============================================
# FUNCIÓN PARA CALCULAR LAS 5 MÉTRICAS
# ============================================
def calcular_metricas_completas(y_true, y_pred, X_test):
    """
    Calcula MAE, RMSE, MAPE, R² y R² Ajustado
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    # R² Ajustado
    n = len(y_true)
    p = X_test.shape[1]
    r2_ajustado = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    
    return {
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE': round(mape, 2),
        'R2': round(r2, 4),
        'R2_Ajustado': round(r2_ajustado, 4)
    }

In [31]:
# Preparar features
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['dia'] = df['fecha'].dt.day

for lag in [1, 2, 3, 7]:
    df[f'lag_{lag}'] = df['demanda_real'].shift(lag)

df['media_movil_7'] = df['demanda_real'].rolling(7).mean()
df = df.dropna().reset_index(drop=True)

feature_cols = ['dia_semana', 'mes', 'dia', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'media_movil_7']
X = df[feature_cols].values
y = df['demanda_real'].values

# División train/test (80/20 respetando orden)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {len(X_train)} días")
print(f"Test: {len(X_test)} días")

Train: 572 días
Test: 144 días


In [32]:
# ============================================
# ENTRENAMIENTO DE LOS 3 MODELOS
# ============================================
print("\n" + "=" * 60)
print(" ENTRENANDO MODELOS (Random Forest, Gradient Boosting, XGBoost)")
print("=" * 60)

resultados = []


 ENTRENANDO MODELOS (Random Forest, Gradient Boosting, XGBoost)


In [33]:
# 1. RANDOM FOREST
print("=" * 60)
print("1. RANDOM FOREST")
print("=" * 60)

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
metrics_rf = calcular_metricas_completas(y_test, y_pred_rf, X_test)

print(f"MAE: {metrics_rf['MAE']} | RMSE: {metrics_rf['RMSE']} | MAPE: {metrics_rf['MAPE']}%")
print(f"R²: {metrics_rf['R2']} | R² Ajustado: {metrics_rf['R2_Ajustado']}")

1. RANDOM FOREST
MAE: 22.16 | RMSE: 28.43 | MAPE: 11.33%
R²: 0.3856 | R² Ajustado: 0.3492


In [34]:
# 2. GRADIENT BOOSTING
print("=" * 60)
print("2. GRADIENT BOOSTING")
print("=" * 60)

gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
metrics_gb = calcular_metricas_completas(y_test, y_pred_gb, X_test)
metrics_gb['Modelo'] = 'Gradient Boosting'

print(f"MAE: {metrics_gb['MAE']} | RMSE: {metrics_gb['RMSE']} | MAPE: {metrics_gb['MAPE']}%")
print(f"R²: {metrics_gb['R2']} | R² Ajustado: {metrics_gb['R2_Ajustado']}")

2. GRADIENT BOOSTING
MAE: 22.13 | RMSE: 28.07 | MAPE: 11.35%
R²: 0.4012 | R² Ajustado: 0.3658


In [35]:
# 3. XGBOOST
print("=" * 60)
print("3. XGBOOST")
print("=" * 60)

xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
metrics_xgb = calcular_metricas_completas(y_test, y_pred_xgb, X_test)
metrics_xgb['Modelo'] = 'XGBoost'

print(f"MAE: {metrics_xgb['MAE']} | RMSE: {metrics_xgb['RMSE']} | MAPE: {metrics_xgb['MAPE']}%")
print(f"R²: {metrics_xgb['R2']} | R² Ajustado: {metrics_xgb['R2_Ajustado']}")

3. XGBOOST
MAE: 22.56 | RMSE: 29.05 | MAPE: 11.53%
R²: 0.3587 | R² Ajustado: 0.3207


In [36]:
# Crear DataFrame con resultados
resultados = [metrics_rf, metrics_gb, metrics_xgb]
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados[['Modelo', 'MAE', 'RMSE', 'MAPE', 'R2', 'R2_Ajustado']]
df_resultados = df_resultados.sort_values('MAE')

print("\n" + "=" * 70)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 70)
print(df_resultados.to_string(index=False))


TABLA COMPARATIVA DE MODELOS
           Modelo   MAE  RMSE  MAPE     R2  R2_Ajustado
Gradient Boosting 22.13 28.07 11.35 0.4012       0.3658
              NaN 22.16 28.43 11.33 0.3856       0.3492
          XGBoost 22.56 29.05 11.53 0.3587       0.3207


In [38]:
import os
os.makedirs('../reports/tables', exist_ok=True)
df_resultados.to_csv('../reports/tables/model_comparison_full.csv', index=False)
print("\n Resultados guardados en reports/tables/model_comparison_full.csv")


 Resultados guardados en reports/tables/model_comparison_full.csv
